In [6]:

from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [7]:
# Trained adapter path
peft_model_id = "/home/jovyan/Works/MSc_Thesis/mh_conversation/finetuned_model/llama3.2_3b_mh_conversation_v2" 

print("Loading model...")

# Load PEFT config to get base model info
config = PeftConfig.from_pretrained(peft_model_id)
print(f"Base model: {config.base_model_name_or_path}")

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map={"": 1},
    torch_dtype=torch.bfloat16
)

# Load tokenizer from the adapter (contains special tokens)
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Resize model to accommodate new tokens
model.resize_token_embeddings(len(tokenizer))

# Load the adapter weights
model = PeftModel.from_pretrained(model, peft_model_id)
model.to(torch.bfloat16)
model.eval()

print("Model loaded successfully!")

Loading model...
Base model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.66it/s]


Tokenizer vocab size: 128256
Model loaded successfully!


In [13]:
# Test the model with the CORRECT tokenizer (the one with special tokens)
def test_model(patient_id, query, max_length=1024):
    # Format exactly like training data - include the explicit instruction
    messages = [
        {
            "role": "system",
            "content": """You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. The assistant gives helpful, comprehensive, and appropriate answers to the user's questions."""
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # Apply chat template using the LOADED tokenizer
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs['input_ids'].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            # temperature=0.1,              
            # top_p=0.95,                 
            # repetition_penalty=1.0,     
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            
            
        )
    
    response_tokens = outputs[0][input_length:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

# Test with the loaded model and tokenizer
test_query = "I\'ve been experiencing extreme levels of stress mainly related to financial difficulties. The constant worry and fear about not being able to make ends meet have led to sleepless nights and increased anxiety. It would be helpful in therapy to explore ways to reduce stress and gain control over my finances."
response = test_model("UID_123", test_query)
print(f"Response: \n{response}")
print("="*80)

Response: 
It's understandable that you're feeling overwhelmed by the stress related to your financial difficulties. It can be incredibly challenging to constantly worry about making ends meet and the impact it has on your sleep and anxiety levels. I want to assure you that you're not alone in this, and there are steps you can take to reduce stress and gain control over your finances.

Firstly, it might be helpful to create a budget that accurately reflects your income and expenses. Start by listing all your sources of income and then categorize your expenses into essential and non-essential ones. This will give you a clearer picture of where your money is going and help you identify areas where you can make adjustments.

Consider reaching out for support from local community resources or organizations that offer financial assistance programs. They may be able to provide guidance or even assistance with managing debt or finding employment opportunities. Additionally, exploring online r